In [19]:
import getopt
import sys
import os
import re
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.backend_bases
import numpy as np
from matplotlib.widgets import Slider, RadioButtons, RectangleSelector, CheckButtons
import h5py as h5
import datetime as dt

In [2]:
frame=['014A_04920_161613', '014A_05232_242525','021D_05266_252525', '116A_05207_252525']

cumfile=f'/work/scratch-pw2/licsar/mnergiz/{frame[0]}/TS_GEOCml10/cum.h5'
ncfile=f'/work/scratch-pw2/licsar/mnergiz/{frame[0]}/{frame[0]}.nc'

In [4]:
cumh5 = h5.File(cumfile,'r')

In [5]:
cumh5

<HDF5 file "cum.h5" (mode r)>

In [8]:
correction_flag=True
vel = cumh5['vel']
try:
    cum = cumh5['cum']
except:
    cum = cumh5['cum_model']
    print('loading model cum data')

n_im, length, width = cum.shape

try:
    gap = cumh5['gap']
    label_gap = 'Gap of ifg network'
except:
    gap = []
    print('No gap field found in {}. Skip.'.format(cumfile))

try:
    geocod_flag = True
    lat1 = float(cumh5['corner_lat'][()])
    lon1 = float(cumh5['corner_lon'][()])
    dlat = float(cumh5['post_lat'][()])
    dlon = float(cumh5['post_lon'][()])
    aspect = np.abs(dlat/dlon/np.cos(np.deg2rad(lat1+dlat*length/2)))
except:
    geocod_flag = False
    aspect = 1
    print('No latlon field found in {}. Skip.'.format(cumfile))

### Add the corrections
if correction_flag:
    # tide correction
    try:
        tide = cumh5['tide'][:]
        label_tide = 'SET correction'
        print('Tide correction found.')
    except KeyError:
        tide = None
        print('No tide correction found in {}. Skip.'.format(cumfile))

    # iono correction
    try:
        iono = cumh5['iono'][:]
        label_iono = 'Iono correction'
        print('Ionospheric correction found.')
    except KeyError:
        iono = None
        print('No iono correction found in {}. Skip.'.format(cumfile))




### Set initial ref area
refarea = cumh5['refarea'][()]
if type(refarea) is bytes:
    refarea = refarea.decode('utf-8')
refx1, refx2, refy1, refy2 = [int(s) for s in re.split('[:/]', refarea)]

print(refarea)
refx1h = refx1-0.5; refx2h = refx2-0.5 ## Shift half for plot
refy1h = refy1-0.5; refy2h = refy2-0.5
point_x = int((refx2+refx1)/2)
point_y = int((refy2+refy1)/2)

Tide correction found.
Ionospheric correction found.
160:161/235:236


In [14]:
mdate = []
### Set master (reference) date
imdates = cumh5['imdates'][()].astype(str).tolist()
if not mdate: ## Not specified or no mdate found in imdates
    ix_m = 0
elif not mdate in imdates:
    print('No {} found in dates. Set reference to {}'.format(mdate, imdates[0]))
    ix_m = 0

cum_ref = cum[ix_m, :, :]

In [17]:
#%% Read Mask (1: unmask, 0: mask, nan: no cum data)
mask_base = np.ones((length, width), dtype=np.float32)
mask_base[np.isnan(cum[ix_m, :, :])] = np.nan

if maskflag:
    print('Reading {} as mask'.format(os.path.relpath(maskfile)))
    mask_vel = io_lib.read_img(maskfile, length, width)

    mask_vel[mask_vel==0] = np.nan ## 0->nan
    mask = mask_vel
else:
    mask = mask_base


In [23]:
#%% Calc time in datetime and ordinal
imdates_dt = np.array(([dt.datetime.strptime(imd, '%Y%m%d') for imd in imdates])) ##datetime
imdates_ordinal = np.array(([dt.datetime.strptime(imd, '%Y%m%d').toordinal() for imd in imdates])) ##73????
imdates_ordinal = imdates_ordinal + mdates.date2num(np.datetime64('0000-12-31'))
mask_base = np.ones((length, width), dtype=np.float32)
mask_base[np.isnan(cum[ix_m, :, :])] = np.nan
mask = mask_base

In [27]:
auto_crange = 99.0
dmin = None
dmax = None
vmin = None
vmax = None
#%% Set color range for displacement and vel
refvalue_lastcum = np.nanmean((cum[-1, refy1:refy2, refx1:refx2]
                               -cum_ref[refy1:refy2, refx1:refx2])*mask[refy1:refy2, refx1:refx2])
dmin_auto = np.nanpercentile((cum[-1, :, :]-cum_ref)*mask, 100-auto_crange)
dmax_auto = np.nanpercentile((cum[-1, :, :]-cum_ref)*mask, auto_crange)
if dmin is None and dmax is None: ## auto
    climauto = True
    dmin = dmin_auto - refvalue_lastcum
    dmax = dmax_auto - refvalue_lastcum
else:
    climauto = False
    if dmin is None: dmin = dmin_auto - refvalue_lastcum
    if dmax is None: dmax = dmax_auto - refvalue_lastcum

refvalue_vel = np.nanmean((vel*mask)[refy1:refy2+1, refx1:refx2+1])
vmin_auto = np.nanpercentile(vel*mask, 100-auto_crange)
vmax_auto = np.nanpercentile(vel*mask, auto_crange)
if vmin is None and vmax is None: ## auto
    vlimauto = True
    vmin = vmin_auto - refvalue_vel
    vmax = vmax_auto - refvalue_vel
else:
    vlimauto = False
    if vmin is None:
        vmin = vmin_auto - refvalue_vel
    if vmax is None:
        vmax = vmax_auto - refvalue_vel

